# 🔥 Active Fire 预测可视化

从所有训练好的 run 中加载最佳模型（run20, val IoU=0.6654），
在训练集上选出 IoU 最高的 sample，展示连续 5 个时间步：
- **上行**：Ground Truth（AF mask）
- **下行**：模型预测

In [ ]:
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import yaml
from pathlib import Path

# 项目根目录
ROOT = Path(r"C:\Users\22125\Desktop\FireST-GraphNet")
sys.path.insert(0, str(ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. 加载所有 run 的配置与最佳 IoU

In [ ]:
import pandas as pd

output_dir = ROOT / "outputs" / "output-clean"
runs = sorted([d for d in output_dir.iterdir() if d.is_dir() and d.name.startswith("run")])

summary = []
for run_dir in runs:
    metrics_f = run_dir / "metrics.csv"
    config_f = run_dir / "config.yaml"
    best_f = run_dir / "best_model.pt"
    if not metrics_f.exists() or not best_f.exists():
        continue
    df = pd.read_csv(metrics_f)
    best_idx = df["val_iou"].idxmax()
    best_iou = df.loc[best_idx, "val_iou"]
    best_f1 = df.loc[best_idx, "val_f1"] if "val_f1" in df.columns else None
    best_epoch = int(df.loc[best_idx, "epoch"])
    with open(config_f) as f:
        cfg = yaml.safe_load(f)
    horizon = cfg.get("forecast", {}).get("horizon", "?")
    summary.append({
        "run": run_dir.name,
        "horizon": horizon,
        "best_epoch": best_epoch,
        "val_iou": best_iou,
        "val_f1": best_f1,
        "path": str(run_dir),
    })

summary_df = pd.DataFrame(summary).sort_values("val_iou", ascending=False)
print(summary_df[["run", "horizon", "best_epoch", "val_iou", "val_f1"]].to_string(index=False))
print(f"\n🏆 最佳 run: {summary_df.iloc[0]['run']}  val_iou={summary_df.iloc[0]['val_iou']:.4f}")

## 2. 加载最佳模型

In [ ]:
from firest_graphnet.models.firest_graphnet import FireSTGraphNet

# 使用最佳 run
best_run = summary_df.iloc[0]
run_dir = Path(best_run["path"])

with open(run_dir / "config.yaml") as f:
    config = yaml.safe_load(f)

mc = config["model"]
fc = config["forecast"]

model = FireSTGraphNet(
    in_channels=mc["in_channels"],
    encoder_channels=mc["encoder_channels"],
    use_cbam=mc["use_cbam"],
    patch_size=mc["patch_size"],
    k_neighbors=mc["k_neighbors"],
    gnn_hidden=mc["gnn_hidden"],
    gnn_layers=mc["gnn_layers"],
    lstm_hidden=mc["lstm_hidden"],
    lstm_layers=mc["lstm_layers"],
    lstm_dropout=mc["lstm_dropout"],
    decoder_channels=mc["decoder_channels"],
    out_channels=mc["out_channels"],
    img_h=config["data"]["img_size"],
    img_w=config["data"]["img_size"],
    forecast_mode=True,
    future_channels=fc["future_channels"],
    future_emb_dim=fc["future_emb_dim"],
)

ckpt = torch.load(run_dir / "best_model.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(device).eval()

horizon = fc["horizon"]
T_in = fc["T_in"]
print(f"✅ 模型加载完成: {best_run['run']}  epoch={ckpt['epoch']}")
print(f"   T_in={T_in}, horizon={horizon}")

## 3. 构建训练数据集 DataLoader

In [ ]:
from firest_graphnet.data.dataset_forecast import TSSatFireForecastDataset
from firest_graphnet.data.firepred_provider import FirePredProvider
from firest_graphnet.data.static_provider import LULCStaticProvider
from torch.utils.data import DataLoader

train_dir = ROOT / "dataset_train"

firepred_prov = FirePredProvider(train_dir / "firepred_train.npy")
lulc_prov = LULCStaticProvider(train_dir / "lulc_train.npy")

train_ds = TSSatFireForecastDataset(
    data_dir=str(train_dir),
    T_in=T_in,
    horizon=horizon,
    label_channel=1,
    firepred_provider=firepred_prov,
    static_provider=lulc_prov,
)

print(f"✅ 训练集样本数: {len(train_ds)}")

## 4. 扫描所有样本，按 IoU 排序找最佳 sample

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(train_ds, batch_size=1, shuffle=False, num_workers=0)

VIS_STEPS = 5  # 可视化的时间步数

sample_scores = []  # (idx, mean_iou_over_5_steps)

print(f"正在扫描 {len(train_ds)} 个样本（取前 {VIS_STEPS} 个时间步的平均 IoU）...")

with torch.no_grad():
    for i, batch in enumerate(loader):
        af_hist = batch["af_hist"].to(device)    # (1, T_in, 8, 256, 256)
        aux_hist = batch["aux_hist"].to(device)  # (1, T_in, 19, 256, 256)
        env_fut = batch["env_fut"].to(device)    # (1, H, 4, 256, 256)
        static = batch["static"].to(device)      # (1, 1, 256, 256)
        af_fut = batch["af_fut"]                 # (1, H, 1, 256, 256)

        logits = model(
            af_hist,
            aux_hist=aux_hist,
            env_fut=env_fut,
            env_source="firepred",
            static=static,
        )  # (1, H, 1, 256, 256)

        preds = (torch.sigmoid(logits) >= 0.5).float().cpu()
        gt = af_fut.float()

        # 计算前 VIS_STEPS 步的平均 IoU
        ious = []
        for t in range(min(VIS_STEPS, horizon)):
            p = preds[0, t, 0]  # (256, 256)
            g = gt[0, t, 0]
            inter = (p * g).sum()
            union = ((p + g) > 0).float().sum()
            iou = (inter / union).item() if union > 0 else 0.0
            ious.append(iou)

        # 只保留有火的 sample（GT 至少有像素）
        gt_sum = gt[0, :VIS_STEPS, 0].sum().item()
        if gt_sum > 50:  # 至少 50 个火像素
            sample_scores.append((i, np.mean(ious), gt_sum))

        if (i + 1) % 50 == 0:
            print(f"  已处理 {i+1}/{len(train_ds)}")

# 按 IoU 降序排列
sample_scores.sort(key=lambda x: x[1], reverse=True)
print(f"\n✅ 扫描完成，{len(sample_scores)} 个有效样本")
print(f"\n🏆 Top-5 样本 (idx, mean_iou, gt_pixels):")
for idx, iou, gtp in sample_scores[:5]:
    print(f"   sample {idx:4d}  IoU={iou:.4f}  GT火像素={gtp:.0f}")

## 5. 可视化最佳样本：GT vs Prediction（5 步）

In [ ]:
# 自定义 colormap: 0=黑色背景, 1=橙红色火
fire_cmap = mcolors.ListedColormap(["#0a0a0a", "#ff4500"])

def visualize_sample(dataset, model, sample_idx, n_steps=5, save_path=None):
    """可视化单个 sample: 上行 GT, 下行 Prediction."""
    batch = dataset[sample_idx]
    af_hist = batch["af_hist"].unsqueeze(0).to(device)
    aux_hist = batch["aux_hist"].unsqueeze(0).to(device)
    env_fut = batch["env_fut"].unsqueeze(0).to(device)
    static = batch["static"].unsqueeze(0).to(device)
    af_fut = batch["af_fut"]  # (H, 1, 256, 256)
    meta = batch["meta"]

    with torch.no_grad():
        logits = model(
            af_hist,
            aux_hist=aux_hist,
            env_fut=env_fut,
            env_source="firepred",
            static=static,
        )
        probs = torch.sigmoid(logits).cpu()
        preds = (probs >= 0.5).float()

    fig, axes = plt.subplots(2, n_steps, figsize=(4 * n_steps, 8))

    for t in range(n_steps):
        gt_t = af_fut[t, 0].numpy()       # (256, 256)
        pred_t = preds[0, t, 0].numpy()    # (256, 256)

        # 计算 IoU
        inter = (gt_t * pred_t).sum()
        union = ((gt_t + pred_t) > 0).sum()
        iou = inter / union if union > 0 else 0.0

        # 上：Ground Truth
        axes[0, t].imshow(gt_t, cmap=fire_cmap, vmin=0, vmax=1, interpolation="nearest")
        axes[0, t].set_title(f"t+{t}", fontsize=14, fontweight="bold")
        axes[0, t].axis("off")

        # 下：Prediction
        axes[1, t].imshow(pred_t, cmap=fire_cmap, vmin=0, vmax=1, interpolation="nearest")
        axes[1, t].set_title(f"IoU={iou:.3f}", fontsize=12)
        axes[1, t].axis("off")

    axes[0, 0].set_ylabel("Ground Truth", fontsize=14, fontweight="bold", labelpad=10)
    axes[1, 0].set_ylabel("Prediction", fontsize=14, fontweight="bold", labelpad=10)
    for ax in axes.flat:
        ax.label_outer()

    event_id = meta.get("event_id", "?")
    fig.suptitle(
        f"Sample #{sample_idx}  |  Event {event_id}  |  {best_run['run']}",
        fontsize=16, fontweight="bold", y=1.02,
    )
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"💾 已保存: {save_path}")

    plt.show()


# 可视化 Top-3 最佳样本
save_dir = ROOT / "outputs" / "qualitative_af_vis"
save_dir.mkdir(parents=True, exist_ok=True)

for rank, (idx, iou, gtp) in enumerate(sample_scores[:3]):
    print(f"\n{'='*60}")
    print(f"Rank #{rank+1}: sample={idx}, mean_IoU={iou:.4f}, GT火像素={gtp:.0f}")
    print(f"{'='*60}")
    visualize_sample(
        train_ds, model, idx, n_steps=VIS_STEPS,
        save_path=str(save_dir / f"best_sample_rank{rank+1}_idx{idx}.png"),
    )

## 6. 概率热力图可视化（连续值）

In [ ]:
def visualize_heatmap(dataset, model, sample_idx, n_steps=5, save_path=None):
    """上行 GT, 下行 sigmoid 概率热力图."""
    batch = dataset[sample_idx]
    af_hist = batch["af_hist"].unsqueeze(0).to(device)
    aux_hist = batch["aux_hist"].unsqueeze(0).to(device)
    env_fut = batch["env_fut"].unsqueeze(0).to(device)
    static = batch["static"].unsqueeze(0).to(device)
    af_fut = batch["af_fut"]
    meta = batch["meta"]

    with torch.no_grad():
        logits = model(
            af_hist,
            aux_hist=aux_hist,
            env_fut=env_fut,
            env_source="firepred",
            static=static,
        )
        probs = torch.sigmoid(logits).cpu()

    fig, axes = plt.subplots(2, n_steps, figsize=(4 * n_steps, 8))

    for t in range(n_steps):
        gt_t = af_fut[t, 0].numpy()
        prob_t = probs[0, t, 0].numpy()

        axes[0, t].imshow(gt_t, cmap=fire_cmap, vmin=0, vmax=1, interpolation="nearest")
        axes[0, t].set_title(f"t+{t}", fontsize=14, fontweight="bold")
        axes[0, t].axis("off")

        im = axes[1, t].imshow(prob_t, cmap="hot", vmin=0, vmax=1, interpolation="nearest")
        axes[1, t].set_title(f"max={prob_t.max():.2f}", fontsize=12)
        axes[1, t].axis("off")

    axes[0, 0].set_ylabel("Ground Truth", fontsize=14, fontweight="bold", labelpad=10)
    axes[1, 0].set_ylabel("Prob (sigmoid)", fontsize=14, fontweight="bold", labelpad=10)

    fig.colorbar(im, ax=axes[1, :], shrink=0.6, label="Fire Probability")

    event_id = meta.get("event_id", "?")
    fig.suptitle(
        f"Probability Heatmap | Sample #{sample_idx} | Event {event_id}",
        fontsize=16, fontweight="bold", y=1.02,
    )
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"💾 已保存: {save_path}")

    plt.show()


# 对最佳样本画概率热力图
best_idx = sample_scores[0][0]
visualize_heatmap(
    train_ds, model, best_idx, n_steps=VIS_STEPS,
    save_path=str(save_dir / f"heatmap_best_idx{best_idx}.png"),
)